# 13 Pooling Metric Comparison

## Purpose
This notebook compares notebook-8 similarity outputs for one fixed checkpoint under three pooling rules: `cls`, `mean`, and `max`.

## Why this notebook matters
This comparison isolates pooling as the only intended change. If the checkpoint, tokenizer family, and notebook-8 run family are held constant, any differences in the distributions or neighborhoods can be attributed to the pooling rule rather than a different model state.

## Inputs
- Three notebook-8 output folders that share the same underlying checkpoint and differ only by pooling rule

## Outputs
- Matched pooling summary tables and merged score tables
- A 3x3 pooling comparison plot matrix
- Top-neighbor overlap summaries across pooling rules
- A self-contained HTML report and an optional clean public-export folder


## Setup note

This notebook reads saved notebook-8 artifacts rather than loading model weights directly:
- code lives in GitHub
- notebook-8 outputs live in Google Drive
- the notebook compares previously saved CSV outputs for matched pooling runs

Because the notebook operates on saved outputs, it can run in a CPU-only Colab session.


## Runtime setup

This cell prepares the Colab runtime by mounting Google Drive, synchronizing the GitHub repository, and making the repository importable.

**Expected output**
- confirmation that Google Drive is mounted
- confirmation that the repository was cloned or updated
- the active repository directory inside the Colab runtime


In [ ]:
# Standard library imports used only for Colab runtime setup.
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', GITHUB_REF],
        check=True,
    )

%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Active repo directory: {REPO_DIR}')


## Import notebook dependencies

This cell imports the shared helper modules used to assemble the pooling comparison tables and reports.

**Expected output**
- no printed output under normal conditions
- a normal Python import error only if the repository sync step failed or a required dependency is missing


In [ ]:
import importlib
from pathlib import Path

from IPython.display import HTML, Image, display

import src.similarity_pooling_comparison as similarity_pooling_comparison
importlib.reload(similarity_pooling_comparison)

from src.notebook_utils import (
    SUPPORTED_TOKENIZER_FAMILIES,
    stringify_path_values,
    validate_tokenizer_family,
    write_json,
)
from src.similarity import (
    build_pooling_metric_comparison,
    build_pooling_run_specs,
    build_public_export_dir,
    build_public_report_subdir,
    export_public_pooling_metric_comparison_html,
)


## User settings

This is the main cell to review before running the notebook. Keep pooling-comparison edits here so the rest of the notebook can remain stable.

**Settings to review**
- `DRIVE_ROOT`: the root Drive folder for the project
- `TOKENIZER_FAMILY`, `EXPERIMENT_NAME`, and `CHECKPOINT_FAMILY`: which saved checkpoint family to compare
- `BASE_OUTPUT_RUN_LABEL`: which notebook-8 run family to inspect
- classifier run labels: which classification checkpoint folders correspond to the checkpoint families
- neighborhood and inspection settings: how the outputs are summarized inline
- public-export settings: whether to package a clean shareable HTML folder

**Expected output**
- this cell only defines settings; it does not run the comparison


In [ ]:
from pathlib import Path

# Update DRIVE_ROOT if the project folder uses a different Google Drive path.
DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# Select the checkpoint family that should be compared across pooling rules.
TOKENIZER_FAMILY = 'manual'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'
CHECKPOINT_FAMILY = 'classification_mlm_init'
BASE_OUTPUT_RUN_LABEL = 'live_extended'

# Keep the saved Drive folder names here even though these classifier runs
# were actually trained with the ep100 setting.
CLASSIFIER_MLM_RUN_LABEL = 'cls_lr2e-5_ep10_bs16_mlm'
CLASSIFIER_RANDOM_RUN_LABEL = 'cls_lr2e-5_ep10_bs16_randominit'

# Configure neighborhood and inspection summaries.
TOP_K_NEIGHBORS = 25
INSPECT_QUERY_ACCESSIONS = []
INSPECT_TOP_N = 12
SCATTER_SAMPLE_SIZE = 20000
HTML_REPORT_TITLE = f'{TOKENIZER_FAMILY} pooling metric comparison'
EMBED_HTML_IMAGES = True

# Configure the optional clean public-export folder.
PUBLIC_EXPORT_ENABLED = True
PUBLIC_EXPORT_PARENT_SUBDIR = 'results/public_reports'
PUBLIC_GITHUB_OWNER = 'hb791-dev'
PUBLIC_GITHUB_REPO = 'glycan-roberta'
PUBLIC_GITHUB_REF = 'main'
PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH = True


## Resolve matched run directories and save the notebook configuration

This cell validates the main categorical settings, builds the three matched notebook-8 run directories, and saves a small JSON configuration record next to the outputs.

**Expected output**
- the resolved pooling-comparison output directory
- one line for each matched pooling run directory
- the public-export destination for a later manual copy step

**How to interpret the result**
- if any matched run folder is missing, review the checkpoint-family or base-output settings before continuing


In [ ]:
validate_tokenizer_family(TOKENIZER_FAMILY, supported_families=SUPPORTED_TOKENIZER_FAMILIES)

CHECKPOINT_FAMILY_CONFIGS = {
    'pretrained_mlm': {
        'checkpoint_source': 'pretraining',
        'model_output_id': 'pretrained_mlm',
        'classifier_run_label': None,
        'display_label': 'Pretrained MLM',
    },
    'classification_mlm_init': {
        'checkpoint_source': 'classification',
        'model_output_id': 'classification_mlm_init',
        'classifier_run_label': CLASSIFIER_MLM_RUN_LABEL,
        'display_label': 'Classifier, MLM init',
    },
    'classification_random_init': {
        'checkpoint_source': 'classification',
        'model_output_id': 'classification_random_init',
        'classifier_run_label': CLASSIFIER_RANDOM_RUN_LABEL,
        'display_label': 'Classifier, random init',
    },
}

if CHECKPOINT_FAMILY not in CHECKPOINT_FAMILY_CONFIGS:
    raise ValueError(
        'Unsupported CHECKPOINT_FAMILY. Choose from: '
        f'{", ".join(sorted(CHECKPOINT_FAMILY_CONFIGS))}'
    )

checkpoint_family_config = CHECKPOINT_FAMILY_CONFIGS[CHECKPOINT_FAMILY]
SIMILARITY_SCALEUP_ROOT = DRIVE_ROOT / 'results' / 'similarity_scaleup'

RUN_SPECS = build_pooling_run_specs(
    scaleup_results_root=SIMILARITY_SCALEUP_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    checkpoint_source=checkpoint_family_config['checkpoint_source'],
    model_output_id=checkpoint_family_config['model_output_id'],
    base_output_run_label=BASE_OUTPUT_RUN_LABEL,
    classifier_run_label=checkpoint_family_config['classifier_run_label'],
)

for spec in RUN_SPECS:
    spec['model_label'] = (
        f"{checkpoint_family_config['display_label']} | {spec['pooling_strategy'].upper()}"
    )

COMPARISON_RUN_LABEL = f'{CHECKPOINT_FAMILY}__{BASE_OUTPUT_RUN_LABEL}__cls_mean_max'
OUTPUT_DIR = (
    DRIVE_ROOT
    / 'results'
    / 'similarity_model_comparison'
    / TOKENIZER_FAMILY
    / EXPERIMENT_NAME
    / 'pooling_metric_comparison'
    / COMPARISON_RUN_LABEL
)
PUBLIC_REPORT_NOTEBOOK_STEM = '13_pooling_metric_comparison'
PUBLIC_EXPORT_PATH_PARTS = [
    TOKENIZER_FAMILY,
    EXPERIMENT_NAME,
    CHECKPOINT_FAMILY,
    COMPARISON_RUN_LABEL,
]
PUBLIC_EXPORT_PARENT_DIR = DRIVE_ROOT / PUBLIC_EXPORT_PARENT_SUBDIR
PUBLIC_EXPORT_REPO_SUBDIR = build_public_report_subdir(
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
PUBLIC_EXPORT_DIR = build_public_export_dir(
    PUBLIC_EXPORT_PARENT_DIR,
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)

comparison_config = {
    'drive_root': str(DRIVE_ROOT),
    'tokenizer_family': TOKENIZER_FAMILY,
    'experiment_name': EXPERIMENT_NAME,
    'checkpoint_family': CHECKPOINT_FAMILY,
    'checkpoint_display_label': checkpoint_family_config['display_label'],
    'base_output_run_label': BASE_OUTPUT_RUN_LABEL,
    'comparison_run_label': COMPARISON_RUN_LABEL,
    'output_dir': str(OUTPUT_DIR),
    'top_k_neighbors': TOP_K_NEIGHBORS,
    'inspect_query_accessions': INSPECT_QUERY_ACCESSIONS,
    'inspect_top_n': INSPECT_TOP_N,
    'scatter_sample_size': SCATTER_SAMPLE_SIZE,
    'html_report_title': HTML_REPORT_TITLE,
    'embed_html_images': EMBED_HTML_IMAGES,
    'run_specs': [stringify_path_values(spec) for spec in RUN_SPECS],
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_json(OUTPUT_DIR / 'pooling_metric_comparison_config.json', comparison_config)

print(f'Comparison output dir: {OUTPUT_DIR}')
print(f'Checkpoint family: {CHECKPOINT_FAMILY} -> {checkpoint_family_config["display_label"]}')
print(f'Base notebook-8 run label stem: {BASE_OUTPUT_RUN_LABEL}')
print('Matched notebook-8 run folders:')
for spec in RUN_SPECS:
    run_dir = Path(spec['run_dir'])
    print(
        f'- {spec["pooling_strategy"].upper()}: {run_dir} | exists={run_dir.exists()} | '
        f'ranked_csv_exists={(run_dir / "specific_vs_all_ranked.csv").exists()}'
    )
print(f'Public export enabled: {PUBLIC_EXPORT_ENABLED}')
print(f'Public export Drive folder: {PUBLIC_EXPORT_DIR}')
print(f'Repo destination after copy: {PUBLIC_EXPORT_REPO_SUBDIR}')


## Build the matched pooling comparison

This is the main workflow cell. It reads the matched notebook-8 outputs, verifies that the runs share the same underlying checkpoint, writes the pooled comparison tables and plots, and optionally packages a clean public-export folder.

**Expected output**
- a list of saved comparison tables and plots
- the shared checkpoint directory used by all three pooling runs
- a link to the full HTML report

**How to interpret the result**
- if the helper raises a matched-pooling error, the three notebook-8 runs are not aligned closely enough to support a fair pooling comparison
- dependency issues or sensitive-string matches should be resolved before any manual copy into the GitHub repository


In [ ]:
comparison_outputs = build_pooling_metric_comparison(
    run_specs=RUN_SPECS,
    output_dir=OUTPUT_DIR,
    report_title=HTML_REPORT_TITLE,
    top_k_neighbors=TOP_K_NEIGHBORS,
    inspect_query_accessions=INSPECT_QUERY_ACCESSIONS,
    inspect_top_n=INSPECT_TOP_N,
    scatter_sample_size=SCATTER_SAMPLE_SIZE,
    embed_html_images=EMBED_HTML_IMAGES,
)

comparison_tables = comparison_outputs['tables']
matched_outputs = comparison_outputs['matched_outputs']
shared_model_dir = matched_outputs['shared_model_dir']

print(f'Matched shared checkpoint folder: {shared_model_dir}')
print('')
print('Saved comparison tables:')
for name, path in comparison_outputs['table_paths'].items():
    print(f'- {name}: {path}')

print('Saved comparison plots:')
for name, path in comparison_outputs['plot_paths'].items():
    print(f'- {name}: {path}')

print(f'Manifest: {comparison_outputs["manifest_path"]}')
print(f'HTML report: {comparison_outputs["plot_paths"]["html_report_path"]}')

display(HTML(
    f'<p><a href="{comparison_outputs["plot_paths"]["html_report_path"]}" target="_blank">Open full HTML pooling-comparison report</a></p>'
))

public_export_artifacts = None

if PUBLIC_EXPORT_ENABLED:
    public_export_artifacts = export_public_pooling_metric_comparison_html(
        comparison_outputs=comparison_outputs,
        export_dir=PUBLIC_EXPORT_DIR,
        repo_public_subdir=PUBLIC_EXPORT_REPO_SUBDIR,
        repo_owner=PUBLIC_GITHUB_OWNER,
        repo_name=PUBLIC_GITHUB_REPO,
        repo_ref=PUBLIC_GITHUB_REF,
    )

    print(f'Public export Drive folder: {public_export_artifacts["public_export_dir"]}')
    print(f'Repo folder to copy into before push: {PUBLIC_EXPORT_REPO_SUBDIR}')
    print(f'Repo report path after push: {public_export_artifacts["repo_index_path"]}')
    print(f'GitHack URL after push: {public_export_artifacts["githack_url"]}')
    print('')

    print('=== Copied public files ===')
    display(public_export_artifacts['copied_files_df'])

    print('=== Dependency issues ===')
    if public_export_artifacts['dependency_issues_df'].empty:
        print('No missing local HTML dependencies were found in the public export.')
    else:
        display(public_export_artifacts['dependency_issues_df'])

    print('=== Sensitive-string scan ===')
    if public_export_artifacts['scan_results_df'].empty:
        print('No obvious personal paths or local-environment strings were found in the copied files.')
    else:
        display(public_export_artifacts['scan_results_df'])

    if public_export_artifacts['has_dependency_issues']:
        raise ValueError(
            'The public export still has missing local dependencies. Fix those before any GitHub copy step.'
        )

    if public_export_artifacts['has_sensitive_matches'] and PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH:
        raise ValueError(
            'The public export still contains suspicious local strings. Review the scan table before sharing the files.'
        )
else:
    print('PUBLIC_EXPORT_ENABLED is False, so the notebook skipped the clean public-export step.')


## Confirm the matched run summary

This table is the first place to look before interpreting any pooling result. It confirms that the three runs point back to the same checkpoint family and use matching query and corpus tables.

**Expected output**
- one summary row per pooling rule showing the aligned run metadata

**How to interpret the result**
- mismatched model directories, query counts, or corpus counts indicate that the comparison is not a valid pooling-only experiment


In [ ]:
display(comparison_tables['matched_pooling_run_summary'])


## Review the global pooling behavior

This cell provides the main global comparison view by summarizing the matched score distributions and displaying the 3x3 pooling matrix.

**Expected output**
- the overall score summary table across pooling rules
- the pairwise score-correlation table
- the saved 3x3 pooling comparison plot

**How to interpret the result**
- the summary table shows broad distribution shifts
- the correlation table shows how closely the pairwise rankings align across pooling rules
- the 3x3 figure makes it easier to judge both distribution shape and pairwise agreement at a glance


In [ ]:
display(comparison_tables['pooling_similarity_summary'])
display(comparison_tables['pooling_pairwise_correlations'])
display(Image(filename=comparison_outputs['plot_paths']['pooling_matrix_plot']))


## Review matched top-neighbor overlap

This cell compares the top-`k` neighborhoods returned for the same query glycans under each pooling rule.

**Expected output**
- a per-query overlap table across pooling pairs
- an averaged overlap summary across pooling pairs
- a three-way overlap summary across `cls`, `mean`, and `max`

**How to interpret the result**
- low overlap means the neighborhood membership changes materially when pooling changes, even though the checkpoint stays fixed


In [ ]:
display(comparison_tables['pooling_top_k_overlap_by_query'])
display(comparison_tables['pooling_top_k_overlap_summary'])
display(comparison_tables['pooling_top_k_three_way_overlap'])


## Review per-query summaries and inspection rows

This cell provides a lightweight follow-up view for individual query glycans after the global pooling summaries have been reviewed.

**Expected output**
- a per-query summary table across pooling rules
- an inspection table for the requested follow-up query glycans

**How to interpret the result**
- use this section when a specific query glycan appears interesting in the global summaries and needs a closer side-by-side review


In [ ]:
display(comparison_tables['pooling_similarity_summary_by_query'])
display(comparison_tables['pooling_query_inspection'])


## Saved outputs

This final cell prints the saved table paths, plot paths, and manifest path so the comparison artifacts can be located quickly after the run finishes.

**Expected output**
- one path listing for each saved table and plot plus the manifest path


In [ ]:
print('Saved table outputs:')
for label, path in comparison_outputs['table_paths'].items():
    print(f'- {label}: {path}')

print('Saved plot outputs:')
for label, path in comparison_outputs['plot_paths'].items():
    print(f'- {label}: {path}')

print(f'Manifest: {comparison_outputs["manifest_path"]}')
